# 04_silver_fhir
Explodes the FHIR R4 bundles in `health.bronze.fhir_bundles` into one row per resource, then flattens the four resource types the analytics need.

| Output table | Grain |
| --- | --- |
| `health.silver.fhir_resources` | one row per FHIR resource (raw JSON kept) |
| `health.silver.fhir_patients` | one row per patient |
| `health.silver.fhir_encounters` | one row per encounter |
| `health.silver.fhir_conditions` | one row per condition (diagnosis) |
| `health.silver.fhir_organizations` | one row per facility |

**Privacy:** as in the HL7 notebook, only name, date of birth, sex and ZIP are kept for matching. SSN, driver's licence, passport, phone and street address are not extracted.

In [0]:
%sql
SET TIME ZONE 'UTC';

## Step 1: one row per resource
`from_json` with the resource typed as `STRING` keeps each resource's JSON intact, whatever its type. Each resource is then parsed only once, by the table that needs it.

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.fhir_resources
COMMENT 'One row per FHIR R4 resource from the Synthea bundles, raw JSON kept. Source for the flattened FHIR tables.'
AS
SELECT
  get_json_object(e.resource, '$.resourceType') AS resource_type,
  get_json_object(e.resource, '$.id')           AS resource_id,
  e.resource                                    AS resource_json,
  b.source_file,
  b.ingested_at
FROM health.bronze.fhir_bundles b
LATERAL VIEW explode(from_json(b.bundle_json, 'entry ARRAY<STRUCT<resource: STRING>>').entry) t AS e;

-- Expect 98,573 resources in total
SELECT resource_type, COUNT(*) AS resources
FROM health.silver.fhir_resources
GROUP BY resource_type ORDER BY resources DESC;

## Step 2: patients

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.fhir_patients
COMMENT 'One row per FHIR Patient. Matching attributes only: no SSN, licence, passport, phone or street address.'
AS
WITH p AS (
  SELECT
    resource_id                                                AS patient_id,
    get_json_object(resource_json, '$.name[0].family')         AS family_name,
    get_json_object(resource_json, '$.name[0].given[0]')       AS given_name,
    get_json_object(resource_json, '$.birthDate')              AS birth_date_raw,
    get_json_object(resource_json, '$.gender')                 AS gender,
    get_json_object(resource_json, '$.address[0].city')        AS city,
    get_json_object(resource_json, '$.address[0].state')       AS state,
    get_json_object(resource_json, '$.address[0].postalCode')  AS postal_code,
    get_json_object(resource_json, '$.deceasedDateTime')       AS deceased_raw
  FROM health.silver.fhir_resources
  WHERE resource_type = 'Patient'
)
SELECT
  patient_id,
  family_name,
  given_name,
  lower(regexp_replace(family_name, '[^A-Za-z]', '')) AS family_norm,
  lower(regexp_replace(given_name,  '[^A-Za-z]', '')) AS given_norm,
  to_date(birth_date_raw)                             AS birth_date,
  CASE gender WHEN 'male' THEN 'M' WHEN 'female' THEN 'F' ELSE 'U' END AS sex,
  city, state,
  left(postal_code, 5)                                AS zip,
  CAST(deceased_raw AS TIMESTAMP)                     AS deceased_ts
FROM p;

## Step 3: encounters
`subject.reference` looks like `urn:uuid:<patient id>`; `serviceProvider.reference` looks like `Organization?identifier=<system>|<organization id>`.

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.fhir_encounters
COMMENT 'One row per FHIR Encounter, with patient and facility keys extracted from references.'
AS
SELECT
  resource_id                                                                      AS encounter_id,
  regexp_extract(get_json_object(resource_json, '$.subject.reference'), 'urn:uuid:(.+)', 1) AS patient_id,
  get_json_object(resource_json, '$.status')                                       AS status,
  get_json_object(resource_json, '$.class.code')                                   AS class_code,   -- AMB, EMER, IMP, HH, VR
  get_json_object(resource_json, '$.type[0].coding[0].code')                       AS type_code,
  get_json_object(resource_json, '$.type[0].coding[0].display')                    AS type_display,
  get_json_object(resource_json, '$.reasonCode[0].coding[0].code')                 AS reason_code,
  get_json_object(resource_json, '$.reasonCode[0].coding[0].display')              AS reason_display,
  CAST(get_json_object(resource_json, '$.period.start') AS TIMESTAMP)              AS start_ts,
  CAST(get_json_object(resource_json, '$.period.end')   AS TIMESTAMP)              AS end_ts,
  regexp_extract(get_json_object(resource_json, '$.serviceProvider.reference'), '\\|(.+)$', 1) AS organization_id,
  get_json_object(resource_json, '$.serviceProvider.display')                      AS facility_name
FROM health.silver.fhir_resources
WHERE resource_type = 'Encounter';

## Step 4: conditions (diagnoses)

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.fhir_conditions
COMMENT 'One row per FHIR Condition, linked to its patient and encounter. Codes are SNOMED CT.'
AS
SELECT
  resource_id                                                                        AS condition_id,
  regexp_extract(get_json_object(resource_json, '$.subject.reference'),   'urn:uuid:(.+)', 1) AS patient_id,
  regexp_extract(get_json_object(resource_json, '$.encounter.reference'), 'urn:uuid:(.+)', 1) AS encounter_id,
  get_json_object(resource_json, '$.code.coding[0].code')                            AS code,
  get_json_object(resource_json, '$.code.coding[0].display')                         AS display,
  get_json_object(resource_json, '$.code.coding[0].system')                          AS code_system,
  get_json_object(resource_json, '$.clinicalStatus.coding[0].code')                  AS clinical_status,
  get_json_object(resource_json, '$.category[0].coding[0].code')                     AS category,
  CAST(get_json_object(resource_json, '$.onsetDateTime')     AS TIMESTAMP)           AS onset_ts,
  CAST(get_json_object(resource_json, '$.abatementDateTime') AS TIMESTAMP)           AS abatement_ts
FROM health.silver.fhir_resources
WHERE resource_type = 'Condition';

## Step 5: organizations (facilities)

In [0]:
%sql
CREATE OR REPLACE TABLE health.silver.fhir_organizations
COMMENT 'One row per FHIR Organization (hospitals and clinics) from hospitalInformation.'
AS
SELECT
  resource_id                                                 AS organization_id,
  get_json_object(resource_json, '$.name')                    AS organization_name,
  get_json_object(resource_json, '$.type[0].coding[0].code')  AS organization_type,
  get_json_object(resource_json, '$.address[0].city')         AS city,
  get_json_object(resource_json, '$.address[0].state')        AS state,
  left(get_json_object(resource_json, '$.address[0].postalCode'), 5) AS zip
FROM health.silver.fhir_resources
WHERE resource_type = 'Organization';

## Validate

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM health.silver.fhir_patients)      AS patients,        -- expect 112
  (SELECT COUNT(*) FROM health.silver.fhir_encounters)    AS encounters,      -- expect 5,087
  (SELECT COUNT(*) FROM health.silver.fhir_conditions)    AS conditions,      -- expect 3,418
  (SELECT COUNT(*) FROM health.silver.fhir_organizations) AS organizations,   -- expect 267
  (SELECT COUNT(*) FROM health.silver.fhir_encounters WHERE patient_id = '' OR start_ts IS NULL) AS bad_encounters,          -- expect 0
  (SELECT COUNT(*) FROM health.silver.fhir_encounters e
     LEFT ANTI JOIN health.silver.fhir_organizations o USING (organization_id))                 AS encounters_without_facility

In [0]:
%sql
-- Encounter mix: AMB 4,735 | EMER 198 | IMP 94 | HH 33 | VR 27
SELECT class_code, COUNT(*) AS encounters
FROM health.silver.fhir_encounters
GROUP BY class_code ORDER BY encounters DESC